In [16]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Define the path to the dataset
dataset_path = '/content/drive/MyDrive/Colab Notebooks/MVSA_Single'

# Check if the path exists and list the first few files
if os.path.exists(dataset_path):
    print(f"Successfully accessed: {dataset_path}")
    files = os.listdir(dataset_path)
    print(f"Total items found: {len(files)}")
    print("Sample files:", files[:5])
else:
    print(f"Path not found: {dataset_path}. Please ensure the folder name is correct.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Successfully accessed: /content/drive/MyDrive/Colab Notebooks/MVSA_Single
Total items found: 2
Sample files: ['data', 'labelResultAll.txt']


In [17]:
import pandas as pd
import os
from PIL import Image
import io
import gc
from concurrent.futures import ThreadPoolExecutor

# Paths
labels_file = '/content/drive/MyDrive/Colab Notebooks/MVSA_Single/labelResultAll.txt'
data_dir = '/content/drive/MyDrive/Colab Notebooks/MVSA_Single/data'
# Path ke file yang baru saja disimpan
parquet_path = '/content/drive/MyDrive/Colab Notebooks/MVSA_Single/mvsa_single_processed.parquet'

def process_single_row(row):
    """Helper function to process one image-text pair with resizing."""
    try:
        item_id = str(int(row['id']))
        image_path = os.path.join(data_dir, f"{item_id}.jpg")
        text_path = os.path.join(data_dir, f"{item_id}.txt")

        t_label = str(row['text']).strip()
        i_label = str(row['image']).strip()

        # Sentiment logic
        if t_label == "neutral":
            f_label = i_label
        elif i_label == 'neutral':
            f_label = t_label
        elif t_label == i_label:
            f_label = t_label
        else:
            return None

        if os.path.exists(image_path) and os.path.exists(text_path):
            # Process and Resize Image to 384x384 (standard for ViLT)
            with Image.open(image_path) as img:
                img = img.convert('RGB')
                img = img.resize((384, 384), Image.Resampling.LANCZOS)
                buf = io.BytesIO()
                img.save(buf, format='PNG')
                img_bytes = buf.getvalue()

            # Process Text
            with open(text_path, 'r', encoding='latin-1') as file:
                text_content = file.read().strip()

            return {
                'ID': item_id,
                'text_sentiment': t_label,
                'image_sentiment': i_label,
                'final_sentiment': f_label,
                'image_bytes': img_bytes,
                'text_content': text_content
            }
    except Exception as e:
        print(f"Error processing row: {e}")
        return None
    return None

# Cek apakah file parquet sudah ada
if os.path.exists(parquet_path):
    print(f"File parquet sudah ada, memuat dari: {parquet_path}")
    df = pd.read_parquet(parquet_path)
    print(f"Total data dimuat: {len(df)}")
    print("\nDistribusi sentimen:")
    print(df['final_sentiment'].value_counts())
    display(df.head())
else:
    print(f"File parquet tidak ditemukan, memulai pemrosesan...")
    print(f"Membaca labels dari: {labels_file}")

    # 1. Read labels
    df_labels = pd.read_csv(labels_file, sep=',', header=0)
    print(f"Total labels dibaca: {len(df_labels)}")
    print(f"Nama kolom dalam file labels: {list(df_labels.columns)}")
    print(f"Sample data:\n{df_labels.head()}")

    # 2. Parallel processing
    print("\nMemulai pemrosesan paralel dengan resizing gambar ke 384x384...")
    rows = [row for _, row in df_labels.iterrows()]

    with ThreadPoolExecutor(max_workers=4) as executor:
        results = list(executor.map(process_single_row, rows))

    # 3. Filter results and cleanup
    processed_records = [r for r in results if r is not None]
    print(f"Total records berhasil diproses: {len(processed_records)}")

    if len(processed_records) > 0:
        df = pd.DataFrame(processed_records)

        # 4. Simpan ke parquet
        print(f"Menyimpan data ke: {parquet_path}")
        df.to_parquet(parquet_path, index=False)
        print("Data berhasil disimpan!")

        del results, processed_records, df_labels, rows
        gc.collect()

        print(f"\nTotal data berhasil diproses dan disimpan: {len(df)}")
        print("\nDistribusi sentimen:")
        print(df['final_sentiment'].value_counts())
        display(df.head())
    else:
        print("PERINGATAN: Tidak ada data yang berhasil diproses!")
        print("Periksa apakah:")
        print(f"  1. File label ada di: {labels_file}")
        print(f"  2. Folder data ada di: {data_dir}")
        print(f"  3. File .jpg dan .txt ada di dalam folder data")

File parquet tidak ditemukan, memulai pemrosesan...
Membaca labels dari: /content/drive/MyDrive/Colab Notebooks/MVSA_Single/labelResultAll.txt
Total labels dibaca: 4869
Nama kolom dalam file labels: ['id', 'text', 'image']
Sample data:
   id      text     image
0   1   neutral  positive
1   2   neutral  positive
2   3   neutral  positive
3   4  positive  positive
4   5  positive  positive

Memulai pemrosesan paralel dengan resizing gambar ke 384x384...
Total records berhasil diproses: 4511
Menyimpan data ke: /content/drive/MyDrive/Colab Notebooks/MVSA_Single/mvsa_single_processed.parquet
Data berhasil disimpan!

Total data berhasil diproses dan disimpan: 4511

Distribusi sentimen:
final_sentiment
positive    2683
negative    1358
neutral      470
Name: count, dtype: int64


,ID,text_sentiment,image_sentiment,final_sentiment,image_bytes,text_content
0,1,neutral,positive,positive,b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\...,How I feel today #legday #jelly #aching #gym
1,2,neutral,positive,positive,b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\...,grattis min griskulting!!!???? va bara tvungen...
2,3,neutral,positive,positive,b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\...,RT @polynminion: The moment I found my favouri...
3,4,positive,positive,positive,b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\...,#escort We have a young and energetic team and...
4,5,positive,positive,positive,b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\...,"RT @chrisashaffer: Went to SSC today to be a ""..."
